# High-Performance Pandas: eval() and query()

## Motivating query and eval: Compound Expressions

In [2]:
import numpy as np

In [8]:
rng = np.random.default_rng(42)
x = rng.random(1000000)
y = rng.random(1000000)
%timeit x + y

214 μs ± 13 μs per loop (mean ± std. dev. of 7 runs, 1,000 loops each)


In [9]:
%timeit np.fromiter((xi + yi for xi, yi in zip(x, y)), dtype=x.dtype, count=len(x))

62.6 ms ± 1.58 ms per loop (mean ± std. dev. of 7 runs, 10 loops each)


In [11]:
mask = (x > 0.5) & (y < 0.5)
mask

array([False, False,  True, ...,  True,  True,  True], shape=(1000000,))

In [12]:
tmp1 = (x > 0.5)
tmp2 = (y < 0.5)
mask = tmp1 & tmp2

In [14]:
!pip3 install numexpr


[notice] A new release of pip is available: 26.1.2 -> 26.2
[notice] To update, run: pip3 install --upgrade pip


In [19]:
import numexpr
import pandas as pd

In [16]:
# using numexpr for faster computation
%timeit numexpr.evaluate('x + y')

224 μs ± 40.7 μs per loop (mean ± std. dev. of 7 runs, 1,000 loops each)


In [17]:
mask_expr = numexpr.evaluate('(x > 0.5) & (y < 0.5)')

In [18]:
np.all(mask == mask_expr)

np.True_

## pandas.eval() for Efficient Operations

In [ ]:
nrows, ncols = 100000, 100
df1, df2, df3, df4 = (pd.DataFrame(rng.random((nrows, ncols)))
                      for i in range(4))


In [23]:
%timeit df1 + df2 + df3 + df4

10.1 ms ± 239 μs per loop (mean ± std. dev. of 7 runs, 100 loops each)


In [24]:
%timeit pd.eval(df1 + df2 + df3 + df4)

10.6 ms ± 95.6 μs per loop (mean ± std. dev. of 7 runs, 100 loops each)


In [25]:
df1, df2, df3, df4, df5 = (pd.DataFrame(rng.integers(0, 1000, (100, 3)))
                          for i in range(5))

### Arithmetic Operations

In [27]:
result1 = -df1 * df2 /(df3 + df4) - df5
result1

,0,1,2
0,-986.037249,-1006.643154,-527.188406
1,-989.469853,-709.805124,-1049.348950
2,-207.556485,-215.034771,-49.923077
3,-1070.408784,-418.561008,-805.391566
4,-675.540380,-4220.385214,-243.002946
...,...,...,...
95,-411.407255,-598.936634,-876.177632
96,-334.504542,-1020.980440,-1379.612583
97,-1471.272861,-1047.350254,-1090.579208
98,-439.860197,-383.001143,-198.370178


In [28]:
result2 = pd.eval(' -df1 * df2 /(df3 + df4) - df5')
np.allclose(result1, result2)

True